## Pattern Pattern1 Index Building Content

This notebook demonstrates how to process documents and build a vector store index for RAG applications. It covers document discovery, text extraction, chunking, and uploading embeddings to a vector database using OGX.

### &#x1F4CB; Contents 
This notebook contains the following sections:

- **[Setup](#Setup)**
  - [Install packages](#Install-packages)
  - [Import required libraries](#Import-required-libraries)
  - [Configure S3 credentials](#Configure-S3-credentials)
  - [Prepare S3 client](#Prepare-S3-client)
- **[Process input documents](#Process-input-documents)**
  - [Documents discovery](#Documents-discovery)
  - [Text extraction](#Text-extraction)
- **[Upload documents content into vector store](#Upload-documents-content-into-vector-store)**
  - [Prepare OGX Client](#Prepare-OGX-Client)
  - [Prepare chunker](#Prepare-chunker)
  - [Initialize vector store](#Initialize-vector-store)
  - [Upload chunks to vector store](#Upload-chunks-to-vector-store)
  - [Retrieve chunks for sample question](#Retrieve-chunks-for-sample-question)
- **[Summary](#Summary)**

---

## Setup

This section sets up the notebook environment by installing required packages, importing libraries, and configuring access to S3 storage.

### Install packages

Install all required Python packages for document processing and RAG operations:
- **ai4rag**: The AutoRAG framework for building RAG applications

In [ ]:
%pip install 'ai4rag~=0.9.3' | tail -n 1

### Import required libraries

Import all necessary Python modules and configure logging to suppress verbose output from component loggers.

In [ ]:
import getpass
import json
import logging
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

for logger_name in (
    "httpx",
    "documents-discovery",
    "text-extraction",
):
    logging.getLogger(logger_name).propagate = False

### Configure S3 credentials

To load documents from S3-compatible object storage, you need to provide credentials. If you're using OpenShift AI, these can be configured as data connections.

&#x1F4CC; **Action**: Provide the credentials for your S3 instance if they are not already set in the notebook environment.

&#x1F4A1; **Tip**: In the project, open **Connections** and add an **S3 compatible object storage connection** to a bucket you will use for documents and test data. Open **Workbenches**, edit your workbench, and attach the S3 connection you created so the notebook can read from the bucket. Save and restart the workbench if prompted.

In [ ]:
required_vars = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_S3_BUCKET"]
missing = [var for var in required_vars if not os.environ.get(var)]
if missing:
    raise ValueError(f"Missing required environment variables: {missing}")

### Prepare S3 client

Creates an S3 client session using the provided credentials. This client will be used to discover and download documents from the specified S3 bucket.

In [ ]:
from ai4rag.components.utils import create_s3_client

try:
    s3_client = create_s3_client()
    s3_client.head_bucket(Bucket=os.environ["AWS_S3_BUCKET"])
except Exception as exc:
    from botocore.exceptions import SSLError

    if isinstance(exc, SSLError):
        warnings.warn("SSL verification failed for S3 — retrying with verify=False.")
        s3_client = create_s3_client(verify=False)
    else:
        raise

---

## Process input documents

This section handles document discovery and text extraction. Documents are first discovered in S3 storage, then their content is extracted and saved as DoclingDocument JSON files for further processing.

The data processing pipeline prepares documents for the RAG system in multiple steps. Each step produces outputs stored under `step_outputs/`. 

| Step | Function | Purpose |
|------|----------|---------| 
| 1 | **`discover_documents`** | List documents in the bucket, prioritize benchmark-referenced docs, apply a size cap, and write a JSON manifest (no content download). |
| 2 | **`extract_text`** | Download the listed documents from S3 and extract text to DoclingDocument JSON files using Docling. |

In [ ]:
from ai4rag.components.data.documents_discovery import discover_documents
from ai4rag.components.data.text_extraction import extract_text

step_output_dir = Path("./step_outputs")
input_data_bucket_name = os.environ["AWS_S3_BUCKET"]
input_data_key = "autorag input data/pdf/bank_policies_pdf/documents"
step_output_dir.mkdir(parents=True, exist_ok=True)

### Documents discovery

Lists objects in the S3 input bucket, filters by supported extensions (e.g., `.pdf`, `.docx`, `.pptx`, `.md`, `.html`, `.txt`), and builds a document set. Documents referenced in the benchmark are prioritized, then others are added until a configurable size limit (1 GB by default) is reached. This step does not download document contents but writes a JSON manifest (`documents_descriptor.json`) containing the bucket, prefix, and list of selected object keys and sizes for the next step.

In [ ]:
result = discover_documents(
    bucket_name=input_data_bucket_name,
    prefix=input_data_key,
    s3_client=s3_client,
)
result.save(step_output_dir / "discovered_documents")

print(json.dumps(result.to_dict(), indent=4, ensure_ascii=False))

### Text extraction

Reads the `documents_descriptor.json` produced by the discovery step, downloads each listed document from S3 into a temporary directory, and runs **Docling** to extract text. Output is one DoclingDocument JSON file per document (e.g., `report.pdf.json`) written to the artifact output path. Each document's `name` field preserves the original filename including its suffix. These files form the final text corpus for the RAG system.

In [ ]:
extracted_text_dir = step_output_dir / "extracted_text"

extraction_result = extract_text(
    documents=result.to_dict()["documents"],
    bucket=result.bucket,
    output_dir=extracted_text_dir,
)

print(
    f"Extracted {extraction_result.processed_count}/{extraction_result.total_documents} documents "
    f"({extraction_result.error_count} errors)"
)

---

## Upload documents content into vector store

This section configures the vector store, chunks the extracted documents, and uploads embeddings to the database for semantic search.

### Prepare OGX Client

The OGX client provides the interface to the embedding models and vector database. This section initializes the client using API credentials from environment variables or prompts.

**Prerequisites:**
- `OGX_CLIENT_API_KEY`: Your authentication key for the OGX API
- `OGX_CLIENT_BASE_URL`: The base URL of your OGX instance (pre-filled from the experiment run; override via environment variable if needed)

&#x1F4A1; **Tip**: In OpenShift AI Workbench, you can add these as environment variables or data connections to avoid entering them manually each time.

In [ ]:
from ai4rag.components.utils import create_ogx_client

OGX_CLIENT_BASE_URL = os.getenv("OGX_CLIENT_BASE_URL", "https://llama-stack-service-llama-stack.apps.rhoai-autorag-pool-gzkqj.aws.rh-ods.com")
OGX_CLIENT_API_KEY = os.getenv("OGX_CLIENT_API_KEY") or getpass.getpass("Please enter 'OGX_CLIENT_API_KEY': ")

client = create_ogx_client(OGX_CLIENT_BASE_URL, OGX_CLIENT_API_KEY)

### Prepare chunker

The chunker splits extracted documents into smaller chunks for more effective retrieval. Configuration includes:
- **Chunking Method**: The algorithm used to split text (e.g., recursive character splitting)
- **Chunk Size**: Maximum number of characters per chunk
- **Chunk Overlap**: Number of overlapping characters between consecutive chunks to preserve context

Proper chunking ensures that retrieved context is both relevant and fits within the model's context window.

In [ ]:
from ai4rag.rag.chunking import LangChainChunker, DoclingChunker

chunking_method = "recursive"
chunk_size = 512
chunk_overlap = 32

if chunking_method == "hybrid":
    chunker = DoclingChunker(max_tokens=chunk_size)
else:
    chunker = LangChainChunker(method=chunking_method, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

### Initialize vector store

The vector store manages document embeddings and enables semantic search. This section configures:
- **Embedding Model**: Converts text chunks into vector representations
- **Vector Database Provider**: The backend storage system (e.g., Milvus)
- **Collection Name**: A named collection where embeddings are stored

The vector store is initialized and ready to receive document chunks.

In [ ]:
from ai4rag.rag.embedding.ogx import OGXEmbeddingModel, OGXEmbeddingParams
from ai4rag.rag.vector_store.ogx import OGXVectorStore

embedding_model_id = "vllm-embedding/ibm-granite/granite-embedding-english-r2"
params = OGXEmbeddingParams(**{'embedding_dimension': 768, 'context_length': 8129})

embedding_model = OGXEmbeddingModel(client=client, model_id=embedding_model_id, params=params)

provider_id = "milvus"
collection_name = "vs_8a3b896e-c894-4796-8a20-1c81a7786667"

ogx_vector_store = OGXVectorStore(
    embedding_model=embedding_model,
    client=client,
    provider_id=provider_id,
    reuse_collection_name=collection_name,
)

### Upload chunks to vector store

This section processes each extracted DoclingDocument JSON file by:
- Loading the DoclingDocument from its JSON representation
- Splitting it into chunks using the configured chunker
- Generating embeddings and uploading them to the vector store

Once complete, all document chunks are indexed and ready for semantic search queries.

In [ ]:
from ai4rag.components.utils import load_docling_documents

documents = load_docling_documents(extracted_text_dir)

for document in documents:
    chunked_documents = chunker.split_documents([document])
    ogx_vector_store.add_documents(chunked_documents)

### Retrieve chunks for sample question

This section demonstrates how to perform a semantic search query against the populated vector store. You can test retrieval by searching for relevant chunks based on a sample question.

In [ ]:
from pprint import pprint

sample_question = input()

results = ogx_vector_store.search(query=sample_question, k=5)
for result in results:
    if isinstance(result, tuple):
        pprint(result[0].model_dump(mode="python"), indent=4)
        continue
    pprint(result.text)

---

## Summary

This notebook successfully processed documents from S3 storage, extracted their text content using Docling, chunked the text into manageable pieces, and uploaded the embeddings to a vector store. The indexed documents are now ready for semantic search and retrieval in RAG applications.